# Model Building - ShipmentSure

This notebook trains machine learning models to predict on-time delivery.

**Models trained:**
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier

**Target variable:** `Reached.on.Time_Y.N` (0 = Late, 1 = On-Time)

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
print("All libraries imported!")

## 1. Load and Prepare Data

In [ ]:
# Load the dataset
df = pd.read_csv('Train.csv')

# Drop ID column
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['Reached.on.Time_Y.N'].value_counts())
df.head()

In [ ]:
# Encode categorical columns
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print("Encoding categorical columns:")
for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    print(f"  ✅ {col} encoded")

df.head()

## 2. Split into Training and Testing Sets

In [ ]:
# Define features (X) and target (y)
X = df.drop('Reached.on.Time_Y.N', axis=1)
y = df['Reached.on.Time_Y.N']

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"\nFeature names: {X.columns.tolist()}")

## 3. Model Training & Evaluation

We train 3 different models and compare their performance.

In [ ]:
# Helper function to train and evaluate a model
def train_and_evaluate(model, model_name, X_train, X_test, y_train, y_test):
    """Train a model and print evaluation metrics."""
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    
    print("="*60)
    print(f"  {model_name}")
    print("="*60)
    print(f"\n  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)\n")
    print("  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Late (0)', 'On-Time (1)']))
    
    return accuracy, y_pred

print("Helper function defined!")

### 3.1 Logistic Regression

A simple linear model that's good for binary classification problems.

In [ ]:
# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_acc, lr_pred = train_and_evaluate(lr_model, 'LOGISTIC REGRESSION', 
                                      X_train, X_test, y_train, y_test)

### 3.2 Decision Tree Classifier

A tree-based model that splits data based on feature thresholds.

In [ ]:
# Model 2: Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_acc, dt_pred = train_and_evaluate(dt_model, 'DECISION TREE', 
                                      X_train, X_test, y_train, y_test)

### 3.3 Random Forest Classifier

An ensemble of decision trees that typically gives better accuracy.

In [ ]:
# Model 3: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_acc, rf_pred = train_and_evaluate(rf_model, 'RANDOM FOREST', 
                                      X_train, X_test, y_train, y_test)

## 4. Model Comparison

In [ ]:
# Compare all models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [lr_acc, dt_acc, rf_acc]
})
results = results.sort_values('Accuracy', ascending=False)

print("\n" + "="*50)
print("  MODEL COMPARISON")
print("="*50)
for _, row in results.iterrows():
    bar = '█' * int(row['Accuracy'] * 50)
    print(f"  {row['Model']:25s} | {row['Accuracy']:.4f} | {bar}")

best_model = results.iloc[0]
print(f"\n  🏆 Best Model: {best_model['Model']} ({best_model['Accuracy']*100:.2f}%)")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(results['Model'], results['Accuracy'] * 100, color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, acc in zip(bars, results['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{acc*100:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Baseline (50%)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Confusion Matrices

In [ ]:
# Plot confusion matrices for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

preds = [lr_pred, dt_pred, rf_pred]
names = ['Logistic Regression', 'Decision Tree', 'Random Forest']

for idx, (pred, name) in enumerate(zip(preds, names)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Late', 'On-Time'], yticklabels=['Late', 'On-Time'])
    axes[idx].set_title(name, fontsize=11)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 6. Feature Importance (Random Forest)

In [ ]:
# Feature importances from Random Forest
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], 
         color='#3498db', edgecolor='black', linewidth=0.5)
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
top5 = feature_importance.tail(5).iloc[::-1]
for _, row in top5.iterrows():
    print(f"  {row['Feature']:25s} : {row['Importance']:.4f}")

## Summary

| Model | Purpose | Key Parameter |
|-------|---------|---------------|
| Logistic Regression | Simple linear baseline | max_iter=1000 |
| Decision Tree | Interpretable tree model | max_depth=5 |
| Random Forest | Ensemble for better accuracy | n_estimators=100, max_depth=5 |

**Conclusion:** The comparison shows which model works best for predicting on-time delivery. Feature importance from Random Forest reveals which factors matter most.